# ASOS Dataset Benchmark (EarlySign V1)

This notebook demonstrates how to use the V1 framework for both live experimental orchestration and historical backtesting using the ASOS Online-Controlled-Experiment dataset.

In [11]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import ibis
from pydantic import BaseModel
from typing import List, Any, Optional

from earlysign.core.ledger import Ledger
from earlysign.v1.methods.group_sequential.protocol_designer import ProtocolDesigner
from earlysign.v1.methods.binomial import BinomialSummaryFact, BatchObservation
from earlysign.v1.templates.binomial_ab import BinomialABTemplate

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Load and Prepare ASOS Data

We use a specific experiment (`3b4300`) and metric from the dataset.

In [12]:
data_path = "data/asos_digital_experiments_dataset.parquet"
if not os.path.exists(data_path):
    !mkdir -p data
    !wget -O {data_path} https://osf.io/62t7f/download

df_raw = pd.read_parquet(data_path)
exp_id = "3b4300"
df = df_raw[
    (df_raw["experiment_id"] == exp_id)
    & (df_raw["metric_id"] == 1)
    & (df_raw["variant_id"] == 1)
].copy()


def get_increments(df: pd.DataFrame):
    df = df.sort_values("time_since_start")
    sc = df["count_c"] * df["mean_c"]
    dn_c = df["count_c"].diff().fillna(df["count_c"]).astype(int)
    ds_c = sc.diff().fillna(sc).astype(int)
    st = df["count_t"] * df["mean_t"]
    dn_t = df["count_t"].diff().fillna(df["count_t"]).astype(int)
    ds_t = st.diff().fillna(st).astype(int)
    return dn_c, ds_c, dn_t, ds_t


dn_c, ds_c, dn_t, ds_t = get_increments(df)
df["dn_c"], df["ds_c"], df["dn_t"], df["ds_t"] = dn_c, ds_c, dn_t, ds_t
df = df[(df["dn_c"] > 0) | (df["dn_t"] > 0)].copy()
df.head()

,experiment_id,variant_id,metric_id,time_since_start,count_c,count_t,mean_c,mean_t,variance_c,variance_t,dn_c,ds_c,dn_t,ds_t
4704,3b4300,1,1,26.5,416826.0,415490.0,0.127703,0.128862,0.111395,0.112257,416826,53230,415490,53541
4705,3b4300,1,1,27.5,431510.0,430232.0,0.129024,0.130062,0.112377,0.113146,14684,2444,14742,2416
4706,3b4300,1,1,28.5,445540.0,444021.0,0.130307,0.131163,0.113327,0.113959,14030,2382,13789,2282
4707,3b4300,1,1,29.5,456786.0,455383.0,0.131372,0.132163,0.114114,0.114696,11246,1952,11362,1946
4708,3b4300,1,1,30.5,470977.0,469357.0,0.131327,0.132113,0.114080,0.114659,14191,1843,13974,1823


## 2. Usage in Practice: Experimental Orchestration

Demonstrating live orchestration with on-demand `progress_report()` calls and the unified `set_protocol` API.

In [13]:
from earlysign.v1.templates.binomial_ab import BinomialABProtocol

con = ibis.duckdb.connect(":memory:")
ledger = Ledger(con, "live_events")
ledger.ensure()

p_control = df.iloc[0]["mean_c"]
planner = ProtocolDesigner()
protocol = planner.plan_binomial_ab(
    alpha=0.05,
    power=0.8,
    p_control=p_control,
    delta=0.005,
    k=3,
)

trial = BinomialABTemplate(ledger)
protocol = BinomialABProtocol(**protocol.model_dump())
trial.set_protocol(protocol)

print(
    f"Design Initialized: n_max={int(protocol.method.efficacy.schedule.interim_points[-1])}"
)

Design Initialized: n_max=144396


In [14]:
for i, (_, row) in enumerate(df.iterrows()):
    batch = []
    if row["dn_c"] > 0:
        batch.append(BatchObservation(n=row["dn_c"], success=row["ds_c"], arm="C"))
    if row["dn_t"] > 0:
        batch.append(BatchObservation(n=row["dn_t"], success=row["ds_t"], arm="T"))

    # update() returns minimal status info
    res = trial.update(batch)

    if res.get("look"):
        # Detailed reporting is done on-demand
        report = trial.report_progress()
        print(f"Look {report['look']} triggered. Status: {report['status']}")
        print(f"  Progress: {report}")

        if res["status"] == "STOP_EFFICACY":
            print("Stopping study early!")
            final = trial.report_result()
            print(f"Final Analysis: {final}")
            break

Look 1 triggered. Status: MONITORING
  Progress: {'look': 1, 'n_c': 416826, 'n_t': 415490, 'z_stat': 1.5811722650328313, 'boundary': 3.499886328027463, 'info_frac': 5.76412088977534, 'status': 'MONITORING'}
Look 2 triggered. Status: MONITORING
  Progress: {'look': 2, 'n_c': 431510, 'n_t': 430232, 'z_stat': 1.4389582660235236, 'boundary': 2.4747933559303044, 'info_frac': 5.967907698274191, 'status': 'MONITORING'}
Look 3 triggered. Status: MONITORING
  Progress: {'look': 3, 'n_c': 445540, 'n_t': 444021, 'z_stat': 1.2001431973297492, 'boundary': 2.0206603136197465, 'info_frac': 6.160565389622981, 'status': 'MONITORING'}


## 3. Historical Analysis: The `run_backtest` API

The `run_backtest` method remains all-in-one, returning a comprehensive `FinalReport` for ease of use.

In [15]:
def batch_stream(df):
    for _, row in df.iterrows():
        batch = []
        if row["dn_c"] > 0:
            batch.append(BatchObservation(n=row["dn_c"], success=row["ds_c"], arm="C"))
        if row["dn_t"] > 0:
            batch.append(BatchObservation(n=row["dn_t"], success=row["ds_t"], arm="T"))
        yield batch


bt_ledger = Ledger(ibis.duckdb.connect(":memory:"), "backtest_events")
bt_ledger.ensure()
bt_trial = BinomialABTemplate(bt_ledger)
bt_trial.set_protocol(protocol)  # Using the already realized protocol

print("Starting historical backtest...")
bt_report = bt_trial.backtest(batch_stream(df))
print(f"Backtest Final Report: {bt_report}")

Starting historical backtest...
Backtest Final Report: {'n_c': 536020, 'n_t': 534896, 'successes_c': 74584, 'successes_t': 75686, 'p_hat_c': 0.13914406178873923, 'p_hat_t': 0.1414966647722174, 'delta_hat': 0.002352602983478169, 'z_stat': 3.504846812334241, 'is_rejected': False, 'final_status': 'CONTINUE'}
